In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/pankrzysiu/cifar10-python/cifar-10-python.tar.gz
/kaggle/input/datasets/pankrzysiu/cifar10-python/cifar-10-batches-py/data_batch_1
/kaggle/input/datasets/pankrzysiu/cifar10-python/cifar-10-batches-py/data_batch_2
/kaggle/input/datasets/pankrzysiu/cifar10-python/cifar-10-batches-py/batches.meta
/kaggle/input/datasets/pankrzysiu/cifar10-python/cifar-10-batches-py/test_batch
/kaggle/input/datasets/pankrzysiu/cifar10-python/cifar-10-batches-py/data_batch_3
/kaggle/input/datasets/pankrzysiu/cifar10-python/cifar-10-batches-py/data_batch_5
/kaggle/input/datasets/pankrzysiu/cifar10-python/cifar-10-batches-py/data_batch_4
/kaggle/input/datasets/pankrzysiu/cifar10-python/cifar-10-batches-py/readme.html


In [2]:
import torch
import torchvision
import torchvision.transforms as transforms

import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import random_split
from torch.utils.data.dataloader import DataLoader

import pandas as pd
import matplotlib.pyplot as plt

import seaborn as sns
import numpy as np
from sklearn.metrics import confusion_matrix

In [3]:
#Taking a part of dataset

train_transform=transforms.Compose([
    transforms.RandomCrop(32,padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465), #Standard for CIFAR-10
        std=(0.2470, 0.2435, 0.2616)
    )
])

test_transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2470, 0.2435, 0.2616)
    )
])

In [4]:
full_train=torchvision.datasets.CIFAR10(
    root="/kaggle/input/datasets/pankrzysiu/cifar10-python",
    train=True,
    download=False,
    transform=train_transform
)

testset=torchvision.datasets.CIFAR10(
    root="/kaggle/input/datasets/pankrzysiu/cifar10-python",
    train=False,
    download=False,
    transform=test_transform
)

In [5]:
generator = torch.Generator().manual_seed(42)

train_subset, val_subset = random_split(
    full_train,
    [45_000, 5_000],
    generator=generator
)

In [6]:
batch_size=128

train_loader=DataLoader(
    train_subset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_subset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    testset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f"Training Sample - {len(train_subset)}")
print(f"Validation Sample - {len(val_subset)}")
print(f"Test Sample - {len(testset)}")

Training Sample - 45000
Validation Sample - 5000
Test Sample - 10000


In [7]:
#Architecture
class CustomCNN(nn.Module):
    def __init__(self,num_classes=10):
        super().__init__()

        self.features=nn.Sequential(

            #1st Block
            nn.Conv2d(3,32,kernel_size=3,padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            #2nd Block
            nn.Conv2d(32,64,kernel_size=3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            #3rd Block
            nn.Conv2d(64,128,kernel_size=3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            #4th Block
            nn.Conv2d(128,256,kernel_size=3,padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),


            nn.AdaptiveAvgPool2d((1,1)) #256x8x8 -> 256x1x1   
        )

        self.classifier=nn.Linear(256,num_classes)


    def forward(self,x):
        x=self.features(x)
        x=torch.flatten(x,1)
        x=self.classifier(x)
        return x

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CustomCNN().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001
)

print("Device:", device)

Device: cuda


In [9]:
import wandb

wandb.login(key="")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 23dcs024 (23dcs024-national-institute-of-technology-hamirpur) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [10]:
wandb.init(
    project="cifar10-cnn-comparison",
    name="cnn_from_scratch",
    config={
        "architecture": "CustomCNN",
        "dataset": "CIFAR-10",
        "epochs": 20,
        "batch_size": 128,
        "learning_rate": 0.001,
        "optimizer": "AdamW"
    }
)

In [11]:
def train_one_epoch(model, loader, criterion, optimizer, device):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_accuracy = correct / total

    return epoch_loss, epoch_accuracy

In [12]:
def validate(model, loader, criterion, device):

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_accuracy = correct / total

    return epoch_loss, epoch_accuracy

In [13]:
num_epochs = 25

train_losses = []
val_losses = []

train_accuracies = []
val_accuracies = []

for epoch in range(num_epochs):

    train_loss, train_accuracy = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    val_loss, val_accuracy = validate(
        model,
        val_loader,
        criterion,
        device
    )

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy
    })

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f}"
    )

Epoch [1/25] Train Loss: 1.3870 | Train Acc: 0.4946 | Val Loss: 1.1642 | Val Acc: 0.5886
Epoch [2/25] Train Loss: 1.0527 | Train Acc: 0.6246 | Val Loss: 1.1095 | Val Acc: 0.6014
Epoch [3/25] Train Loss: 0.9238 | Train Acc: 0.6738 | Val Loss: 1.0344 | Val Acc: 0.6310
Epoch [4/25] Train Loss: 0.8352 | Train Acc: 0.7052 | Val Loss: 0.9631 | Val Acc: 0.6584
Epoch [5/25] Train Loss: 0.7722 | Train Acc: 0.7297 | Val Loss: 0.8397 | Val Acc: 0.7038
Epoch [6/25] Train Loss: 0.7154 | Train Acc: 0.7505 | Val Loss: 0.7584 | Val Acc: 0.7278
Epoch [7/25] Train Loss: 0.6853 | Train Acc: 0.7611 | Val Loss: 0.8186 | Val Acc: 0.7070
Epoch [8/25] Train Loss: 0.6495 | Train Acc: 0.7747 | Val Loss: 0.7844 | Val Acc: 0.7286
Epoch [9/25] Train Loss: 0.6196 | Train Acc: 0.7832 | Val Loss: 0.7079 | Val Acc: 0.7540
Epoch [10/25] Train Loss: 0.5983 | Train Acc: 0.7927 | Val Loss: 0.7701 | Val Acc: 0.7350
Epoch [11/25] Train Loss: 0.5668 | Train Acc: 0.8061 | Val Loss: 0.6472 | Val Acc: 0.7690
Epoch [12/25] Train

In [14]:
model.eval()

test_correct = 0
test_total = 0

all_predictions = []
all_labels = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()

        all_predictions.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_accuracy = test_correct / test_total

print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

Test Accuracy: 0.8082
Test Accuracy: 80.82%


In [15]:
wandb.finish()

epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
train_accuracy,▁▃▄▅▆▆▆▆▇▇▇▇▇▇▇▇█████████
train_loss,█▆▅▄▄▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁
val_accuracy,▁▁▂▃▄▅▅▅▆▅▆▇▇▆▇▇█▇▇█▇████
val_loss,█▇▇▆▄▄▄▄▃▄▂▂▃▃▂▁▁▂▂▁▂▁▁▁▁
epoch,25
train_accuracy,0.85973
train_loss,0.40371
val_accuracy,0.8198
val_loss,0.52967
